# Critical Habitat (English) Exploration

> **Note:** This dataset is delivered as ZIP files (and may include spatial formats), which adds setup and loading complexity that may be beyond our current beginner project needs.

**Objective:** Perform initial inspection and summary of the English critical habitat dataset for Species at Risk in Canada.

**Data source(s):**
- Dataset page: https://open.canada.ca/data/en/dataset/47caa405-be2b-4e9e-8f53-c478ade2ca74
- English ZIP: https://data-donnees.az.ec.gc.ca/api/file?path=%2Fspecies%2Fprotectrestore%2Fcritical-habitat-species-at-risk-canada%2FCriticalHabitat.zip

**Date/author:** 2026-03-11, Peter


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile
import pandas as pd

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False

print(f"GeoPandas available: {HAS_GEOPANDAS}")


In [ ]:
english_zip = "CriticalHabitat.zip"
english_zip_url = "https://data-donnees.az.ec.gc.ca/api/file?path=%2Fspecies%2Fprotectrestore%2Fcritical-habitat-species-at-risk-canada%2FCriticalHabitat.zip"

print(f"Expected ZIP file: {english_zip}")

if not Path(english_zip).exists():
    print("ZIP not found locally. Downloading now...")
    urlretrieve(english_zip_url, english_zip)
    print("Download complete.")
else:
    print("ZIP already exists locally; skipping download.")

print(f"File exists: {Path(english_zip).exists()}")


In [ ]:
zip_listing_df = pd.DataFrame()

if Path(english_zip).exists():
    with zipfile.ZipFile(english_zip, "r") as zf:
        names = zf.namelist()
    zip_listing_df = pd.DataFrame({"archive_path": names})
    zip_listing_df["extension"] = zip_listing_df["archive_path"].str.rsplit(".", n=1).str[-1].str.lower()
    display(zip_listing_df.head(20))
    print(f"Total files in ZIP: {len(zip_listing_df)}")
else:
    print("English ZIP not found. Keep it in the same folder as this notebook and rerun.")


In [ ]:
df = None
gdf = None

try:
    df = pd.read_csv(english_zip, compression="zip")
    print(f"Loaded table rows with pandas: {len(df)}")
    display(df.head())
except Exception as exc:
    print("Pandas could not open a table directly from this ZIP.")
    print(f"Reason: {exc}")
    print("This usually means the ZIP contains spatial files (like .shp or .gpkg) instead of a CSV.")

    if HAS_GEOPANDAS:
        try:
            gdf = gpd.read_file(f"zip://{Path(english_zip).resolve()}")
            print(f"Loaded spatial rows with GeoPandas: {len(gdf)}")
            display(gdf.head())
        except Exception as geo_exc:
            print(f"GeoPandas could not open this ZIP either: {geo_exc}")
    else:
        print("GeoPandas is not installed, so spatial fallback is unavailable.")


In [ ]:
if df is not None:
    summary = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[col].dtype) for col in df.columns],
        "null_count": [int(df[col].isna().sum()) for col in df.columns]
    })
    display(summary)
elif gdf is not None:
    summary = pd.DataFrame({
        "column": gdf.columns,
        "dtype": [str(gdf[col].dtype) for col in gdf.columns],
        "null_count": [int(gdf[col].isna().sum()) for col in gdf.columns]
    })
    display(summary)
else:
    print("No dataset loaded yet. Check ZIP contents and rerun.")


### Next Steps
1. Confirm layer format(s) and projection.
2. Add topic-specific analyses (species counts, province/territory breakdown, and area summaries).
3. Add map visualizations and export cleaned analysis outputs.


---
### Open In Colab
[Open this notebook in Google Colab](https://colab.research.google.com/github/Peter/ICD2O-OSC-2026-Project-Plan-Team-2/blob/main/notebooks/critical-habitat-english-exploration.ipynb)
